# 1. OBJETIVO
1. Analisar os dados dos arquivos json:
* offer; 
* profile ; 
* transactions 
2. Usar o spark 
3. Explorar os dados, relacionamentos
3. Analisar o perfil dos dados para encontrar:
* campos vazios
* outliers ( usar isolation forest )
* contagem de dados
* gráficos

In [0]:
import os
from pyspark.sql import functions as F

# 2. LISTA DE ARQUIVOS

In [0]:
# Lista os arquivos para ter certeza que encontrou o diretório
display(dbutils.fs.ls("/Workspace/Users/marcosferreira220863@gmail.com/ifood_chalenge"))

# 3. DIRETÓRIO DE DADOS DESSE PROJETO

In [0]:
data_path="dbfs:/Workspace/Users/marcosferreira220863@gmail.com/ifood_chalenge/data/raw"

# 4.EXPLORANDO DADOS

## 4.1. Offers.json

In [0]:

file_name = "offers.json"
file_path = os.path.join(data_path,file_name)
df_offer = spark.read.json(file_path)
display(df_offer)


In [0]:
df2=df_offer.select(   #"id"\
                 "discount_value"\
                ,"channels"\
                ,"discount_value"\
                ,"duration"\
                ,"min_value"\
                ,"offer_type"                
                )\
        .groupBy("discount_value"\
                ,"channels"
                ,"duration"\
                ,"min_value"\
                ,"offer_type")\
        .count().alias("contagem")\
        .orderBy("discount_value"\
                ,"channels"\
                ,"duration"\
                ,"offer_type")\
        .filter(F.col("count")>0)
        #.orderBy(F.col("contagem").desc())            

display(df2)

### 4.1.1. Contagem de Linhas

In [0]:
print(f"total de ofertas: {df_offer.count()}")

### 4.1.2. Verificação de Duplicados

In [0]:
df2= df_offer.select("channels","discount_value","duration","min_value","offer_type").orderBy("channels","discount_value","duration","min_value","offer_type")
display(df2)

## 4.2. Profile.json

In [0]:
file_name = "profile.json"
file_path = os.path.join(data_path,file_name)
df = spark.read.json(file_path)
display(df)

Databricks data profile. Run in Databricks to view.

### 4.2.1. Contagem de linhas Profile

#### a. _total de linhas_

In [0]:
print(df.columns    )

In [0]:
print(f"Total de Linhas da tabela profile {df.count()}")

### 4.2.2. Perfil da table profile

#### a. _Contagem de Nulos_

In [0]:
from pyspark.sql import functions as F

def profile_data(df):
    """
    Função para gerar um resumo completo do dataframe:
    - Contagem de nulos por coluna
    - Percentual de nulos
    - Tipos de dados
    - Quantidade de valores únicos
    : param df: Dataframe a ser analisado
    : return: Dataframe com o resumo
    """
    total_rows = df.count()
    
    # Lista de tuplas com: Nome da Coluna, Tipo, Nulos, % Nulos, Valores Distintos
    profile_list = []
    
    for col_name in df.columns:
        null_count = df.filter(F.col(col_name).isNull()).count()
        null_pct = (null_count / total_rows) * 100
        distinct_count = df.select(col_name).distinct().count()
        dtype = dict(df.dtypes)[col_name]
        
        profile_list.append((col_name, dtype, null_count, null_pct, distinct_count))
    
    # Criar um DataFrame com o resumo
    columns = ["Coluna", "Tipo", "Total_Nulos", "Percentual_Nulos", "Valores_Unicos"]
    profile_df = spark.createDataFrame(profile_list, columns)
    
    return profile_df

# Uso da função
df_summary = profile_data(df)
display(df_summary)

#### b. _Distribuição das colunas numéricas _ 

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_histograms(df, columns):
    # Convertemos apenas as colunas necessárias para o driver
    # O sample(0.1) pega 10% dos dados para não sobrecarregar
    #pdf = df.select(columns).sample(0.1).toPandas()
    pdf = df.select(columns).toPandas()
    
    for col in columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(pdf[col], kde=True)
        plt.title(f"Distribuição da coluna: {col}")
        plt.show()

# Lista de colunas numéricas para plotar
num_cols = ["age", "credit_card_limit"]
plot_histograms(df, num_cols)

#### c. _distribuição da coluna de sexo ( não nulas )_

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
def plot_cat_histograms(df):
    """
    Função para plotar histogramas de colunas categóricas.
    : param df: Dataframe a ser analisado
    """
    # Obter colunas categóricas
    cat_cols = [col for (col, dtype) in df.dtypes if( (dtype == "string") and (col) != "id" ) ]
    
    # Plotar histogramas
    plot_histograms(df, cat_cols)
plot_cat_histograms(df)


#### d. _Verficação de MNAR(missing values not at random) entre idade e sexo
O propósito dessa célula é verificar se os valores onde o gênero é nulo coincidem com os valores onde a idade é 118 ( que já um outlier )


In [0]:
from pyspark.sql import functions as F

# 1. Criar uma coluna categórica para o status da idade e gênero
df_flag = df.withColumn("age_status",
    F.when(F.col("age") == 118, "Idade 118")\
     .when(F.col("age").isNull(), "Idade Nula")\
     .otherwise("Idade Válida")
).withColumn("gender_status",\
    F.when(F.col("gender").isNull(), "Gênero Nulo")\
     .otherwise("Gênero Preenchido")
)

# 2. Agrupar para contar as combinações
# Fazemos o .collect() aqui porque o resultado é pequeno (apenas 4 ou 5 combinações)
df_plot = df_flag.groupBy("age_status", "gender_status") \
                 .count() \
                 .toPandas()

print(df_plot)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

# Usamos o dataframe Pandas criado no passo anterior
chart = sns.barplot(
    data=df_plot,
    x="age_status",
    y="count",
    hue="gender_status",
    palette="viridis" # Escolha uma paleta de cores profissional
)

plt.title('Relação entre Idade Suspeita (118) e Gênero Nulo', fontsize=15)
plt.xlabel('Status da Idade', fontsize=12)
plt.ylabel('Quantidade de Clientes', fontsize=12)
plt.legend(title='Status do Gênero')

# Adicionar os valores nas barras (opcional, mas ajuda na apresentação)
for container in chart.containers:
    chart.bar_label(container, fmt='%.0f')

plt.show()

A análise acima mostra que dados que apresentam idade (age )== 118 tem correlação com dados onde o sexo(gender )==null. Esses dados devem ser removidos do modelo e armazenados na área de dados processados.

#### e. _valores máximos e mínimos de variáveis numéricas

In [0]:

from pyspark.sql.types import IntegerType, DoubleType, FloatType, LongType
from pyspark.sql import functions as F

df_2 = df.withColumn("registered_on", F.col("registered_on").cast(LongType()))
for col in df_2.columns:
    if  isinstance(df_2.schema[col].dataType, (IntegerType, DoubleType, FloatType, LongType)):
         stats = df_2.agg(F.min(col).alias('min'), F.max(col).alias('max'))
         print(f"minimo da coluna {col} é {stats.first()['min']} e o máximo é {stats.first()['max']}")

    else:
        print(f"a coluna {col} não é numérica")    

## 4.3. Transactions.json   

### 4.3.1. Contagem de Linhas

In [0]:
import os
data_path="dbfs:/Workspace/Users/marcosferreira220863@gmail.com/ifood_chalenge/data/raw"

file_name = "transactions.json"
file_path = os.path.join(data_path,file_name)
df = spark.read.json(file_path)
df = df.orderBy("account_id","time_since_test_start","event")
display(df)

In [0]:
count = df.count()
print(f"O dataset possui {count:,} linhas")
    

### 4.3.2. Identificando valores distintos para a coluna evento

In [0]:
import pyspark.sql.functions as F

df_distinct = df.select(F.column("event")).distinct()
display(df_distinct)

### 4.3.3. Identificando valores únicos no campo value

In [0]:
df_unique = df.select(F.column("value")).distinct()
display(df_unique)


In [0]:
df_3 = df.select(F.col('event'), F.col('value')).orderBy(F.col('event'))
display(df_3)

In [0]:
df_group = df.groupBy('account_id').agg(F.count('*').alias('count')).orderBy(F.col('count').desc())
display(df_group)                                                

In [0]:
#df_select = df.filter(F.col("account_id").like("%94de646f7b6041228ca7dec82adb97d2%"))
df_select = df.select(F.col('event'), F.col('time_since_test_start'), F.col('value')).filter(F.col("account_id")=="94de646f7b6041228ca7dec82adb97d2").orderBy(F.col('time_since_test_start'))
display(df_select)  

In [0]:
df_offer2 = df_offer.filter(F.col("id")=="9b98b8c7a33c4b65b9aebfe6a799e6d9")
display(df_offer2)